In [1]:
import os

In [2]:
%pwd

'c:\\Users\\sagal\\OneDrive\\Desktop\\Let us build\\emotion_detection\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'c:\\Users\\sagal\\OneDrive\\Desktop\\Let us build\\emotion_detection'

In [6]:
import random
import numpy as np
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report

#prevent duplicate openMP runtime library errors
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

#device check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [10]:
from dataclasses import dataclass
from pathlib import Path
import yaml

In [8]:
#entity first update
@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    batch_size: int
    num_epochs: int
    patience: int
    seed: int
    lr_layer3: float
    lr_layer4: float
    lr_fc: float

In [9]:
from emotion_detection.constant import *
from emotion_detection.utils.common import *

In [11]:
#paths to files
config_filepath = "config/config.yaml"
params_filepath = "params.yaml"

In [21]:
#configuration manager 2nd update

class ConfigurationManager:
    def __init__(self,config_filepath = "config/config.yaml",params_filepath = "params.yaml" ):
        with open(config_filepath, "r") as f:
            self.config = yaml.safe_load(f)
            
        with open(params_filepath, "r") as f:
            self.params = yaml.safe_load(f)

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config_info = self.config['model_trainer']
        params_info = self.params['ModelTrainer']
        
        return ModelTrainerConfig(
            root_dir=Path(config_info['root_dir']),
            train_data_path=Path(config_info['train_data_path']),
            test_data_path=Path(config_info['test_data_path']),
            model_name=config_info['model_name'],
            batch_size=params_info['batch_size'],
            num_epochs=params_info['num_epochs'],
            patience=params_info['patience'],
            seed=params_info['seed'],
            lr_layer3=params_info['lr_layer3'],
            lr_layer4=params_info['lr_layer4'],
            lr_fc=params_info['lr_fc']
        )

In [17]:
#components third update very important update

class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config
        
        #reproducibility
        random.seed(self.config.seed)
        torch.manual_seed(self.config.seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.config.seed)

    def _get_data_loaders(self):
        #image augmentation and normalizations
        train_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.Grayscale(num_output_channels=3),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        val_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        #loading image datasets directly from transformation artifacts
        base_train = datasets.ImageFolder(root=str(self.config.train_data_path), transform=train_transform)
        base_val = datasets.ImageFolder(root=str(self.config.train_data_path), transform=val_transform)
        test_set = datasets.ImageFolder(root=str(self.config.test_data_path), transform=val_transform)

        #85/15 split
        indices = list(range(len(base_train)))
        random.shuffle(indices)
        split = int(0.85 * len(indices))
        
        train_subset = Subset(base_train, indices[:split])
        val_subset = Subset(base_val, indices[split:])

        #calculate inverse frequencey weights to manage class imbalnce
        train_labels = [base_train.targets[i] for i in indices[:split]]
        class_counts = Counter(train_labels)
        total_samples = len(train_labels)
        class_weights = {cls: total_samples / count for cls, count in class_counts.items()}
        sample_weights = [class_weights[label] for label in train_labels]

        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )

        #production data loaders
        train_loader = DataLoader(train_subset, batch_size=self.config.batch_size, sampler=sampler)
        val_loader = DataLoader(val_subset, batch_size=self.config.batch_size, shuffle=False)
        test_loader = DataLoader(test_set, batch_size=self.config.batch_size, shuffle=False)

        return train_loader, val_loader, test_loader, len(train_subset), len(val_subset), test_set.classes

    def _build_model(self, num_classes=7):
        # Load the base network and target specific layers for training
        #load the base network and target specific layers for trainin
        model = models.resnet18(weights="IMAGENET1K_V1")
        for name, param in model.named_parameters():
            if "layer3" in name or "layer4" in name or "fc" in name:
                param.requires_grad = True
            else:
                param.requires_grad = False

        #trail 2 dropout layer
        model.fc = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(model.fc.in_features, num_classes)
        )
        return model.to(device)

    def initiate_model_training(self):
        os.makedirs(self.config.root_dir, exist_ok=True)
        
        train_loader, val_loader, test_loader, train_size, val_size, classes = self._get_data_loaders()
        model = self._build_model(num_classes=len(classes))
        
        criterion = nn.CrossEntropyLoss()
        optimizer = Adam([
            {"params": model.layer3.parameters(), "lr": self.config.lr_layer3},
            {"params": model.layer4.parameters(), "lr": self.config.lr_layer4},
            {"params": model.fc.parameters(),     "lr": self.config.lr_fc}
        ])

        best_val_acc = 0.0
        patience_counter = 0
        save_path = os.path.join(self.config.root_dir, self.config.model_name)
            #training
        for epoch in range(self.config.num_epochs):
            model.train()
            train_loss, train_correct = 0.0, 0
            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
                train_correct += (outputs.argmax(1) == labels).sum().item()

            train_acc = (train_correct / train_size) * 100
            avg_train_loss = train_loss / len(train_loader)

            #validation
            model.eval()
            val_loss, val_correct = 0.0, 0
            with torch.no_grad():
                for images, labels in val_loader:
                    images, labels = images.to(device), labels.to(device)
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    val_loss += loss.item()
                    val_correct += (outputs.argmax(1) == labels).sum().item()

            val_acc = (val_correct / val_size) * 100
            avg_val_loss = val_loss / len(val_loader)

            print(f"Epoch [{epoch+1}/{self.config.num_epochs}] Train Loss: {avg_train_loss:.4f} Acc: {train_acc:.2f}% | Val Loss: {avg_val_loss:.4f} Acc: {val_acc:.2f}%")

            
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                patience_counter = 0
                torch.save(model.state_dict(), save_path)
                print(f"  Superior performance saved to disk ({best_val_acc:.2f}%)")
            else:
                patience_counter += 1
                print(f"  No improvement recorded ({patience_counter}/{self.config.patience})")
                if patience_counter >= self.config.patience:
                    print(f"Early stopping rule executed at epoch {epoch+1}")
                    break

        #out of sample evaluation
        print('/n final test evaluation')
        model.load_state_dict(torch.load(save_path))
        model.eval()
        
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(device)
                outputs = model(images)
                preds = outputs.argmax(1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels.numpy())

        test_acc = np.mean(np.array(all_preds) == np.array(all_labels)) * 100
        print(f"Final Test Accuracy Score: {test_acc:.2f}%\n")
        print(classification_report(all_labels, all_preds, target_names=classes))

In [22]:
try:
    config_manager = ConfigurationManager()
    model_trainer_config = config_manager.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.initiate_model_training()
except Exception as e:
    logger.error(f"Pipeline execution failed: {str(e)}")
    raise e

Epoch [1/40] Train Loss: 1.8391 Acc: 26.82% | Val Loss: 1.7440 Acc: 30.42%
  Superior performance saved to disk (30.42%)
Epoch [2/40] Train Loss: 1.5037 Acc: 42.85% | Val Loss: 1.4676 Acc: 45.14%
  Superior performance saved to disk (45.14%)
Epoch [3/40] Train Loss: 1.3185 Acc: 50.33% | Val Loss: 1.3156 Acc: 51.33%
  Superior performance saved to disk (51.33%)
Epoch [4/40] Train Loss: 1.1873 Acc: 55.59% | Val Loss: 1.1898 Acc: 56.11%
  Superior performance saved to disk (56.11%)
Epoch [5/40] Train Loss: 1.0902 Acc: 59.83% | Val Loss: 1.1443 Acc: 58.61%
  Superior performance saved to disk (58.61%)
Epoch [6/40] Train Loss: 0.9819 Acc: 63.92% | Val Loss: 1.0632 Acc: 61.22%
  Superior performance saved to disk (61.22%)
Epoch [7/40] Train Loss: 0.9202 Acc: 66.34% | Val Loss: 1.0369 Acc: 62.57%
  Superior performance saved to disk (62.57%)
Epoch [8/40] Train Loss: 0.8426 Acc: 69.24% | Val Loss: 1.0149 Acc: 63.82%
  Superior performance saved to disk (63.82%)
Epoch [9/40] Train Loss: 0.8132 